# **Q-02: KIỂM TRA TÍNH HỢP LỆ DỮ LIỆU ESC-50**

## **1. Cài đặt & Import thư viện**

In [8]:
import os
import pandas as pd
import numpy as np
import torchaudio
from pathlib import Path

## **2. Kết nối Drive & Đường dẫn**

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/esc50/data/ESC-50-master")
AUDIO_DIR = DATA_DIR / "audio"
META_PATH = DATA_DIR / "meta" / "esc50.csv"

## **3. Đọc file metadata**

In [9]:
df = pd.read_csv(META_PATH)
print(f"\nĐọc metadata thành công: {len(df)} dòng")
print("\nCác cột trong metadata:")
print(df.columns.tolist())


Đọc metadata thành công: 2000 dòng

Các cột trong metadata:
['filename', 'fold', 'target', 'category', 'esc10', 'src_file', 'take']


## **4. Kiểm tra cơ bản**

### **4.1 Tổng số mẫu**

In [10]:
if AUDIO_DIR.exists():
    audio_files = list(AUDIO_DIR.glob("*.wav"))
    print(f"Số lượng file .wav tìm thấy trong thư mục audio: {len(audio_files)}")
else:
    print(f"Thư mục không tồn tại: {AUDIO_DIR}")

Số lượng file .wav tìm thấy trong thư mục audio: 2000


### **4.2 Số lớp**

In [14]:
n_classes = df['target'].nunique()
print(f"- Số lượng lớp: {n_classes} lớp")

- Số lượng lớp: 50 lớp


### **4.3 Số mẫu mỗi lớp**

In [16]:
samples_per_class = df['category'].value_counts()
print(f"- Số mẫu mỗi lớp: Đều có {samples_per_class.unique()[0]} mẫu/lớp")

- Số mẫu mỗi lớp: Đều có 40 mẫu/lớp


### **4.4 Phân bố theo Fold**

In [17]:
folds = df['fold'].unique()
folds.sort()
print(f"- Danh sách các fold: {folds.tolist()} (Tổng cộng: {len(folds)} folds)")

samples_per_fold = df['fold'].value_counts().sort_index()
print("\nPhân phối số mẫu trên từng fold:")
for fold, count in samples_per_fold.items():
    print(f"  + Fold {fold}: {count} mẫu")

- Danh sách các fold: [1, 2, 3, 4, 5] (Tổng cộng: 5 folds)

Phân phối số mẫu trên từng fold:
  + Fold 1: 400 mẫu
  + Fold 2: 400 mẫu
  + Fold 3: 400 mẫu
  + Fold 4: 400 mẫu
  + Fold 5: 400 mẫu


### **4.5 Kiểm tra file âm thanh tồn tại**

In [21]:
missing_files = []

for filename in df['filename']:
    file_path = AUDIO_DIR / filename
    if not file_path.exists():
        missing_files.append(filename)

if len(missing_files) == 0:
    print("Toàn bộ 2000 file âm thanh được liệt kê trong metadata đều tồn tại đầy đủ trong thư mục audio!")
else:
    print(f"Có {len(missing_files)} file bị thiếu!")
    print("Danh sách file thiếu (tối đa hiển thị 10 file):", missing_files[:10])

Toàn bộ 2000 file âm thanh được liệt kê trong metadata đều tồn tại đầy đủ trong thư mục audio!


### **4.6 Kiểm tra tên file khớp với fold và target**

In [22]:
mismatches = []

for idx, row in df.iterrows():
    filename = row['filename']
    expected_fold = row['fold']
    expected_target = row['target']

    # Tên file có cấu trúc: {fold}-{clip_id}-{take}-{target}.wav
    parts = filename.replace(".wav", "").split("-")

    if len(parts) == 4:
        file_fold = int(parts[0])
        file_target = int(parts[3])

        if file_fold != expected_fold or file_target != expected_target:
            mismatches.append({
                'filename': filename,
                'row_fold': expected_fold,
                'file_fold': file_fold,
                'row_target': expected_target,
                'file_target': file_target
            })
    else:
        mismatches.append({
            'filename': filename,
            'error': "Tên file không đúng định dạng chuẩn {fold}-{clip_id}-{take}-{target}.wav"
        })

if len(mismatches) == 0:
    print("100% tên file khớp hoàn toàn với thông tin 'fold' và 'target' trong metadata!")
else:
    print(f"Phát hiện {len(mismatches)} trường hợp không khớp!")
    mismatches_df = pd.DataFrame(mismatches)
    display(mismatches_df.head(10))

100% tên file khớp hoàn toàn với thông tin 'fold' và 'target' trong metadata!


### **5. Kiểm tra thông số file âm thanh — lấy 5 file ngẫu nhiên**

In [27]:
import random
import wave

random_samples = df.sample(n=5, random_state=42)

info_list = []

for idx, row in random_samples.iterrows():
    file_path = AUDIO_DIR / row['filename']

    try:
        with wave.open(str(file_path), 'rb') as wav_file:
            sample_rate = wav_file.getframerate()
            channels = wav_file.getnchannels()
            num_frames = wav_file.getnframes()
            duration = num_frames / sample_rate

        waveform, _ = torchaudio.load(str(file_path))

        info_list.append({
            "Filename": row['filename'],
            "Category": row['category'],
            "Sample Rate (Hz)": sample_rate,
            "Channels": channels,
            "Num Frames": num_frames,
            "Duration (s)": round(duration, 2),
            "Tensor Shape": list(waveform.shape)
        })
    except Exception as e:
        info_list.append({
            "Filename": row['filename'],
            "Category": row['category'],
            "Error": str(e)
        })

audio_info_df = pd.DataFrame(info_list)
display(audio_info_df)

,Filename,Category,Sample Rate (Hz),Channels,Num Frames,Duration (s),Tensor Shape
0,5-221950-A-22.wav,clapping,44100,1,220500,5.0,"[1, 220500]"
1,1-79220-A-17.wav,pouring_water,44100,1,220500,5.0,"[1, 220500]"
2,4-165845-A-45.wav,train,44100,1,220500,5.0,"[1, 220500]"
3,3-130330-A-22.wav,clapping,44100,1,220500,5.0,"[1, 220500]"
4,4-157297-A-21.wav,sneezing,44100,1,220500,5.0,"[1, 220500]"


### **6. Thống kê số bản ghi gốc theo lớp**

In [31]:
class_counts = df.groupby(['target', 'category']).size().reset_index(name='count')

pd.set_option('display.max_rows', 50)
display(class_counts)

,target,category,count
0,0,dog,40
1,1,rooster,40
2,2,pig,40
3,3,cow,40
4,4,frog,40
5,5,cat,40
6,6,hen,40
7,7,insects,40
8,8,sheep,40
9,9,crow,40
